# 02 · Signal-processing engine
The heart of StyleGAN3-T: Kaiser FIR design, filtered up/downsampling, and the continuous-domain leaky ReLU. **Do not build the generator until these are verified.**

In [ ]:
import os, sys
# ---- platform auto-detect: the same notebook runs on Colab and Kaggle ----
PLATFORM = "kaggle" if os.path.exists("/kaggle/input") else "colab"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if (PLATFORM == "colab" or PLATFORM == "kaggle") and not os.path.exists("src"):
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/Ravikishore710/styleforge3-T.git"], check=True)
    os.chdir("styleforge3-T")
sys.path.insert(0, os.path.abspath("."))
!pip install -q -r requirements.txt
import tensorflow as tf
print("platform:", PLATFORM, "| TF:", tf.__version__,
      "| GPU:", tf.config.list_physical_devices("GPU"))
# Kaggle: enable GPU (Settings -> Accelerator -> GPU P100) and add the FFHQ
# dataset to /kaggle/input, or run scripts/prepare_ffhq.py --source folder.

## Filter design

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.ops.filters import design_kaiser_filter, binomial_filter
f = design_kaiser_filter(cutoff=0.2, half_width=0.05, sampling_rate=1.0)
print('kaiser taps:', len(f), '| DC gain:', f.sum())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(f); ax[0].set_title('Kaiser low-pass kernel'); ax[0].grid(alpha=.3)
w = np.fft.rfft(f, 4096); freqs = np.fft.rfftfreq(4096)
ax[1].plot(freqs, 20 * np.log10(np.abs(w) + 1e-8)); ax[1].set_title('frequency response (dB)')
ax[1].axvline(0.2, color='r', ls='--', label='cutoff'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Filtered resampling vs naive

In [ ]:
import tensorflow as tf
from src.ops.upfirdn import upsample2d
from src.ops.filters import binomial_filter
x = tf.random.normal([1, 64, 64, 1])
up_fine = upsample2d(x, binomial_filter(5))
up_nearest = tf.image.resize(x, [128, 128], method='nearest')
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, im, t in [(ax[0], x[0,...,0], 'input 64px'),
                 (ax[1], up_nearest[0,...,0], 'nearest 2x (spectral images)'),
                 (ax[2], up_fine[0,...,0], 'filtered 2x (alias-free)')]:
    F = np.fft.fftshift(np.abs(np.fft.fft2(im.numpy())))
    a.imshow(np.log1p(F), cmap='magma'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

## Continuous-domain nonlinearity (filtered leaky ReLU)

In [ ]:
from src.ops.filtered_lrelu import filtered_lrelu
from src.ops.filters import design_kaiser_filter
t = np.arange(256, dtype=np.float32)
s = np.sin(2 * np.pi * 0.45 * t).astype(np.float32)
x = tf.constant(s[None, :, None, None] * np.ones([1, 1, 256, 1], dtype=np.float32))
fd = design_kaiser_filter(0.25, 0.05, sampling_rate=1.0)
y_f = filtered_lrelu(x, fu=None, fd=fd, down=2)
y_n = tf.nn.leaky_relu(x, 0.2)[:, ::2, ::2, :]
def hf(img):
    p = np.abs(np.fft.fft2(img[0, ..., 0].numpy() - img[0, ..., 0].numpy().mean())) ** 2
    h = p.shape[0]; m = np.zeros_like(p); m[:h//4] = 1; m[3*h//4:] = 1
    return (p * m).sum() / p.sum()
print(f'Nyquist energy | naive: {hf(y_n):.3f} | filtered: {hf(y_f):.3f}')
assert hf(y_f) < hf(y_n) * 0.5